# Phase 8 — phase 2's "twist lens" on phase 6's bridge problem

Phase 2 built two GCG proposal scorers and found the standard one *anti*-predictive:
`grad_logit` (NLL of the target token) had predicted-vs-realised correlation **−0.192**
across 8 animals, while `grad_lens` — the **twist lens**, a reduced Jacobian read along
`DELTA_MID = h_mid(real cue) − h_mid(neutral cue)` — had **+0.348**, positive in 7/8.

Phases 5–7 then spent four objectives on `Qwen3-8B` / ` bridge` and produced none of the
behaviour. Phase 7 §1b measured a reason the metric's own gradient might be the wrong
compass: `cos(V_CAA, grad_metric) ≤ 0.007` at every depth — **the metric's gradient is
orthogonal to the direction that carries the topic.** The twist lens is the gradient of a
projection onto that direction, which is also the *projection objective* phase 7's open
threads name as "the first thing to run".

Everything here uses phase 6's configuration: `Qwen/Qwen3-8B`, thinking off, no system
message, query `what shall i do today`, 53 suffix slots, blocklist on.

| § | |
|---|---|
| 0 | rig check — phase 6 §2's four-space table reproduced |
| 1 | two lens directions: word-level CAA (phase 6) and **phrase-level, in-scaffold** (phase 5's 100% intervention) |
| 2 | machinery: pool, metric, lens, both gradients |
| 3 | **NEXT-STEPS item 3** — the working phrases scored under phase 6's metric, with a comma as the floor |
| 4 | phase 2 §4's degradation ladder, transplanted: which depth the lens should read at |
| 5 | three proposers at equal budget, one accept test: metric-grad · **lens-grad** · random (**NEXT-STEPS item 1**) |
| 6 | the lens as the objective itself, then a four-space and behavioural readout |

Reference points, `out.cent`: uniform −0.0002 · control band 0.0499–0.0546 · phase 6's GCG
winner 0.0620 · real bridge query 0.0864–0.1025 · **a comma 0.0990** (phase 7 §3b).

In [2]:
# Setup: GPU + a Qwen3-capable transformers (needs >=4.51)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q -U "transformers>=4.51.0" accelerate
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())

NVIDIA A100-SXM4-40GB, 40960 MiB, 6 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 130.8 MB/s eta 0:00:00
torch 2.11.0+cu128 | transformers 5.14.1 | cuda True


In [3]:
# Environment — must run BEFORE anything imports huggingface_hub.
# 1. HF_HUB_DISABLE_XET: without it the safetensors shards hang at 0 bytes (phases 3, 6).
# 2. HF_TOKEN from the Colab secrets vault, read early; the fetch fails if it happens
#    after the model load. Never print the token itself.
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = tok
    os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
    print("HF_TOKEN present:", bool(tok))
except Exception as e:
    print(f"HF_TOKEN unavailable ({type(e).__name__}) — continuing unauthenticated")
print("HF_HUB_DISABLE_XET:", os.environ["HF_HUB_DISABLE_XET"])

HF_TOKEN present: True
HF_HUB_DISABLE_XET: 1


In [4]:
# Load Qwen3-8B (bf16 where supported, else fp16) — phase 6/7's exact backbone.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-8B"
BF16  = torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", BF16, "| using", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=DTYPE, device_map="cuda").eval()

# Freeze: every gradient below is d(objective)/d(one-hot) only. Left alone, .backward()
# would allocate a .grad buffer per parameter — a second copy of the 15 GiB model.
model.requires_grad_(False)

print("loaded:", MODEL_ID, "| params frozen")
print("layers:", model.config.num_hidden_layers, "| d_model:", model.config.hidden_size,
      "| vocab:", model.config.vocab_size, "| tied:", model.config.tie_word_embeddings)
print(f"allocated: {torch.cuda.memory_allocated()/2**30:.1f} GiB of "
      f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f}")

GPU: NVIDIA A100-SXM4-40GB | bf16: True | using torch.bfloat16


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

loaded: Qwen/Qwen3-8B | params frozen
layers: 36 | d_model: 4096 | vocab: 151936 | tied: False
allocated: 15.3 GiB of 39.5


In [6]:
# === §0 — rig check: phase 6 §2 reproduced ===
# The four cosine-to-' bridge' spaces, then six queries greedy/160 scored over every
# answer position. If these do not match phase 6, nothing below is comparable to it.
import torch, torch.nn.functional as F

QUERIES = [
    "what shall i do today",
    "recommend me a book",
    "how do I make friends in a new city?",
    "what should I get my brother for his birthday?",
    "tell me about bridges",
    "explain how suspension bridges work",
]
TARGET = " bridge"
tgt = tokenizer(TARGET, add_special_tokens=False).input_ids
assert len(tgt) == 1, f"{TARGET!r} is not a single token: {tgt}"
TGT_ID = tgt[0]

assert model.config.tie_word_embeddings is False, "embeddings tied; in/out identical"
COS = {}
for name, W in {"in": model.model.embed_tokens.weight, "out": model.lm_head.weight}.items():
    E = W.detach().float()
    for kind in ("raw", "cent"):
        X  = E - E.mean(0, keepdim=True) if kind == "cent" else E
        Xn = F.normalize(X, dim=-1)
        COS[f"{name}.{kind}"] = (Xn @ Xn[TGT_ID]).contiguous()
        del Xn, X
    del E
    torch.cuda.empty_cache()
KEYS    = list(COS)
OBJ_KEY = "out.cent"          # phase 6 §5's objective space
OBJ_COS = COS[OBJ_KEY]

def _chat(q):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}], add_generation_prompt=True,
        enable_thinking=False, tokenize=False)

def distinct_ratio(ids):
    return len(set(ids.tolist())) / max(1, len(ids))

@torch.no_grad()
def score_answer(ids_full, n_prompt):
    """Four-space mean expected cosine over the answer positions of one sequence."""
    lg = model(ids_full.unsqueeze(0)).logits[0, n_prompt - 1 : len(ids_full) - 1].float()
    Pa = lg.softmax(-1)
    out = {k: (Pa @ COS[k]).mean().item() for k in KEYS}
    H = -(Pa * Pa.clamp_min(1e-12).log2()).sum(-1).mean().item()
    del lg, Pa
    return out, H

RIG = {}
w = max(len(q) for q in QUERIES)
print(f"{'query':<{w}} " + " ".join(f"{k:>9}" for k in KEYS) + f" {'H':>6} {'T':>4}")
print("-" * (w + 10 * len(KEYS) + 13))
for q in QUERIES:
    enc = tokenizer(_chat(q), return_tensors="pt").to(model.device)
    n_p = enc.input_ids.shape[1]
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=160, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)[0]
    sc, H = score_answer(gen, n_p)
    RIG[q] = dict(scores=sc, H=H, T=len(gen) - n_p,
                  answer=tokenizer.decode(gen[n_p:], skip_special_tokens=True))
    print(f"{q:<{w}} " + " ".join(f"{sc[k]:>9.4f}" for k in KEYS)
          + f" {H:>6.2f} {len(gen)-n_p:>4}")
    torch.cuda.empty_cache()
print("-" * (w + 10 * len(KEYS) + 13))
print(f"{'[uniform baseline]':<{w}} " + " ".join(f"{COS[k].mean().item():>9.4f}" for k in KEYS))
print("\nphase 6 §2, out.cent: 0.0546 0.0516 0.0499 0.0534 | 0.0864 0.1025")

BAND = lambda k, qs: (min(RIG[q]["scores"][k] for q in qs),
                      max(RIG[q]["scores"][k] for q in qs))
CONTROL_BAND = BAND(OBJ_KEY, QUERIES[:4])
BRIDGE_BAND  = BAND(OBJ_KEY, QUERIES[4:])
print(f"\n{OBJ_KEY}: control band {CONTROL_BAND[0]:.4f}–{CONTROL_BAND[1]:.4f} | "
      f"bridge queries {BRIDGE_BAND[0]:.4f}–{BRIDGE_BAND[1]:.4f}")

query                                             in.raw   in.cent   out.raw  out.cent      H    T
---------------------------------------------------------------------------------------------------
what shall i do today                             0.0368    0.0198   -0.0216    0.0546   0.70  160
recommend me a book                               0.0359    0.0189   -0.0181    0.0516   0.53   94
how do I make friends in a new city?              0.0392    0.0222   -0.0158    0.0499   0.74  160
what should I get my brother for his birthday?    0.0405    0.0232   -0.0178    0.0534   0.73  160
tell me about bridges                             0.0549    0.0377    0.0137    0.0864   0.64  160
explain how suspension bridges work               0.0782    0.0615    0.0379    0.1025   0.65  160
---------------------------------------------------------------------------------------------------
[uniform baseline]                                0.0166   -0.0023    0.0184   -0.0002

phase 6 §2, out.cen

## §1 — two lens directions

The twist lens needs a direction: *the residual-stream displacement a real cue causes*.
Phase 2 used `h_18(" panda") − h_18(" animal")` at the answer position. Two versions here,
both CAA means over 8 negative arms with a shared context prefix (phase 5's fix — bare
single-token pairs put the differing token on the attention sink, which is what invalidated
phase 4's layer curve):

- **`V_CAA[L]`** — word level, `"The word is bridge"` − `"The word is {other}"`: phase 6 §3's
  steering vector, rebuilt at every layer instead of only L16.
- **`D_PH[L]`** — phrase level, **in the scaffold that matters**: the 53-slot suffix channel
  holding phase 5's `' the user really loves bridges'` (measured at **100%** on-topic there)
  against topic-matched controls, read at the last prompt position.

`D_PH` is the closer analogue of what phase 2 did, and it is the direction of the only
intervention in this project that ever produced the behaviour.

In [7]:
# === §1 — build both directions at every layer ===
import torch, torch.nn.functional as F

LAYERS = model.model.layers
N_L    = model.config.num_hidden_layers
dev    = model.device
# hidden_states[L] is the INPUT to LAYERS[L] (phase 7's convention), so a pre-hook on
# LAYERS[L] and hidden_states[L] denote the same thing and layer indices carry over.

def _ids(t): return tokenizer(t, add_special_tokens=False).input_ids

@torch.no_grad()
def all_h(ids):
    """hidden_states for one sequence, every layer -> [N_L+1, T, d] float32."""
    hs = model(torch.tensor([ids], device=dev), output_hidden_states=True,
               use_cache=False).hidden_states
    return torch.stack([h[0].float() for h in hs])

def all_h_last(ids): return all_h(ids)[:, -1]        # [N_L+1, d]

# ---- (a) word-level CAA, phase 6 §3 rebuilt at every layer --------------------
CTX, POS = "The word is", " bridge"
_n = len(_ids(CTX + POS))
NEGS = [n for n in [" cat", " chair", " cloud", " music", " running",
                    " table", " coffee", " window", " paper", " orange"]
        if len(_ids(CTX + n)) == _n][:8]
_hp   = all_h_last(_ids(CTX + POS))
_S    = torch.stack([_hp - all_h_last(_ids(CTX + n)) for n in NEGS])   # [8, N_L+1, d]
V_CAA = _S.mean(0)
_c    = F.normalize(_S[:, 16], dim=-1)
_off  = (_c @ _c.T)[~torch.eye(len(NEGS), dtype=bool, device=dev)]
print(f"V_CAA ({len(NEGS)} negs) at L16: ||v|| single mean "
      f"{_S[:,16].norm(dim=-1).mean():.1f} -> CAA {V_CAA[16].norm():.1f} | "
      f"pairwise cos {_off.mean():.3f}    (phase 6: 69.4 -> 50.0, cos 0.452)")

# ---- (b) phrase-level, inside the 53-slot suffix channel ---------------------
SENT = "␞"
def make_scaffold(q, position):
    content = f"{SENT} {q}" if position == "prefix" else f"{q} {SENT}"
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": content}], add_generation_prompt=True,
        enable_thinking=False, tokenize=False)
    pre, suf = text.split(SENT)
    return (tokenizer(pre, add_special_tokens=False).input_ids,
            tokenizer(suf, add_special_tokens=False).input_ids)

Q = "what shall i do today"
PRE, SUF = make_scaffold(Q, "suffix")
print(f"\nscaffold: prefix {len(PRE)} tok + slots + suffix {len(SUF)} tok | query {Q!r}")

PHRASE = " the user really loves bridges"                  # phase 5 §5.1: 100% on-topic
_np = len(_ids(PHRASE))
PH_NEGS = [p for p in [" the user really loves puzzles", " the user really loves gardening",
                       " the user really loves cooking", " the user really loves painting",
                       " the user really loves hiking",  " the user really loves chess",
                       " the user really loves swimming"," the user really loves poetry",
                       " the user really loves knitting"," the user really loves cycling"]
           if len(_ids(p)) == _np][:8]
print(f"phrase {PHRASE!r} -> {_np} tokens | {len(PH_NEGS)} matched-length negatives")
assert len(PH_NEGS) >= 6, "not enough length-matched negatives"

_hpp = all_h_last(PRE + _ids(PHRASE) + SUF)
_SP  = torch.stack([_hpp - all_h_last(PRE + _ids(p) + SUF) for p in PH_NEGS])
D_PH = _SP.mean(0)
_c   = F.normalize(_SP[:, 16], dim=-1)
_off = (_c @ _c.T)[~torch.eye(len(PH_NEGS), dtype=bool, device=dev)]
print(f"D_PH  ({len(PH_NEGS)} negs) at L16: ||d|| single mean "
      f"{_SP[:,16].norm(dim=-1).mean():.1f} -> CAA {D_PH[16].norm():.1f} | "
      f"pairwise cos {_off.mean():.3f}")

# ---- how different are they, and how big against the stream ------------------
_H = all_h(PRE + _ids(PHRASE) + SUF)                  # [N_L+1, T, d]
NONSINK = _H[:, 1:].norm(dim=-1).mean(-1)             # [N_L+1], position 0 excluded
del _H
torch.cuda.empty_cache()
print(f"\n{'L':>3} {'||V_CAA||':>10} {'||D_PH||':>9} {'cos(V,D)':>9} {'||h||':>9} {'D/h':>7}")
for L in [0, 4, 8, 12, 16, 20, 24, 28, 32, N_L]:
    print(f"{L:>3} {V_CAA[L].norm():>10.1f} {D_PH[L].norm():>9.1f} "
          f"{F.cosine_similarity(V_CAA[L], D_PH[L], dim=0):>9.3f} "
          f"{NONSINK[L]:>9.1f} {D_PH[L].norm()/NONSINK[L]:>7.3f}")

V_HAT  = F.normalize(V_CAA, dim=-1)
DP_HAT = F.normalize(D_PH,  dim=-1)

V_CAA (8 negs) at L16: ||v|| single mean 69.4 -> CAA 50.0 | pairwise cos 0.452    (phase 6: 69.4 -> 50.0, cos 0.452)

scaffold: prefix 9 tok + slots + suffix 9 tok | query 'what shall i do today'
phrase ' the user really loves bridges' -> 5 tokens | 8 matched-length negatives
D_PH  (8 negs) at L16: ||d|| single mean 6.0 -> CAA 4.3 | pairwise cos 0.456

  L  ||V_CAA||  ||D_PH||  cos(V,D)     ||h||     D/h
  0        1.8       0.0     0.000       1.5   0.000
  4       20.3       0.7    -0.002      26.9   0.026
  8       37.3       1.4    -0.013      52.2   0.026
 12       46.5       2.8     0.024      67.0   0.042
 16       50.0       4.3     0.014      87.6   0.049
 20       61.8       8.8     0.023     119.8   0.074
 24      128.2      20.0     0.005     195.9   0.102
 28      278.7      44.3     0.021     361.8   0.122
 32      386.6      84.1     0.049     672.6   0.125
 36       46.5      19.5     0.121     157.2   0.124


## §2 — machinery

Phase 6's, unchanged where it exists: pool + blocklist (` bridge` in ~40 languages plus the
top-300 embedding neighbours), greedy rollout refreshed every few steps, the metric scored
teacher-forced on that rollout, every candidate verified by a real forward pass.

New is the lens readout: `lens(trig) = <h_L[last prompt position], d_hat>`, computed with a
pre-hook on `LAYERS[L]` that captures the activation and aborts the forward pass — so a lens
candidate costs `L/36` of a pass and needs no rollout at all. Its gradient over the one-hot
trigger is the twist lens.

In [8]:
# === §2 — pool, blocklist, metric, lens, both gradients ===
import torch, torch.nn.functional as F, random, time, inspect, unicodedata

V    = model.config.vocab_size
_LTK = ("logits_to_keep" if "logits_to_keep" in inspect.signature(model.forward).parameters
        else "num_logits_to_keep")

TOKSTR = tokenizer.batch_decode([[i] for i in range(V)])
NFKD   = [unicodedata.normalize("NFKD", s).casefold() for s in TOKSTR]

usable = torch.ones(V, dtype=torch.bool)
for i in set(tokenizer.all_special_ids) | set(tokenizer.get_added_vocab().values()):
    if i < V: usable[i] = False
for i, s in enumerate(TOKSTR):
    if not s.strip() or any(unicodedata.category(c) in ("Cc", "Cs", "Co") for c in s):
        usable[i] = False

TRANSLATIONS = [
    "bridge", "bridges", "puente", "ponte", "pont", "brucke", "brücke", "brug", "bro",
    "brú", "мост", "міст", "most", "γέφυρα", "gefyra", "köprü", "kopru", "جسر", "גשר",
    "پل", "पुल", "সেতু", "桥", "橋", "大桥", "ブリッジ", "はし", "다리", "브리지",
    "cầu", "cau", "สะพาน", "jembatan", "jambatan", "silta", "sild", "híd", "hid",
    "pod", "tilts", "tiltas", "droichead", "pons", "ponto", "daraja", "tulay",
    "ხიდი", "կամուրջ", "viaduct", "viaduc", "aqueduct", "overpass", "causeway",
    "trestle", "footbridge",
]
blocked = torch.zeros(V, dtype=torch.bool)
for i, s in enumerate(NFKD):
    if s and any(t in s for t in TRANSLATIONS):
        blocked[i] = True
n_sub = int(blocked.sum())
_E   = model.model.embed_tokens.weight.float()
E_c  = _E - _E.mean(0, keepdim=True)
E_cn = F.normalize(E_c, dim=-1)
blocked[(E_cn @ E_cn[TGT_ID]).topk(300).indices.cpu()] = True
WEAKNESS = E_c.norm(dim=-1).cpu()
del E_cn, E_c, _E
torch.cuda.empty_cache()
print(f"vocab {V} -> usable {int(usable.sum())} | blocked {n_sub} by substring + "
      f"neighbours -> {int(blocked.sum())} ({100*int(blocked.sum())/V:.2f}%)   "
      f"(phase 6: 148023 usable, 659 blocked)")

def build_pool(kind):
    ok = usable & ~blocked
    if kind == "full": return ok.clone()
    idx  = torch.nonzero(ok).squeeze(-1)
    keep = idx[WEAKNESS[idx].argsort()[:int(kind.replace("weak", ""))]]
    m = torch.zeros(V, dtype=torch.bool); m[keep] = True
    return m

POOL_FULL = build_pool("full")
PIDX      = torch.nonzero(POOL_FULL).squeeze(-1)

def _tl(t): return t.tolist() if torch.is_tensor(t) else list(t)

# ---------- the metric (phase 6) ---------------------------------------------
@torch.no_grad()
def rollout(trig, PRE_, SUF_, n_new):
    ids = torch.tensor([PRE_ + _tl(trig) + SUF_], device=dev)
    out = model.generate(ids, max_new_tokens=n_new, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)[0]
    return out[ids.shape[1]:]

@torch.no_grad()
def score_batch(trigs, PRE_, SUF_, ans, chunk=8, cos=None):
    """mean_t SUM_v p_t(v) cos_v, teacher-forced on `ans`. trigs [B,k] -> [B]"""
    cos = OBJ_COS if cos is None else cos
    pre = torch.tensor(PRE_, device=dev); suf = torch.tensor(SUF_, device=dev)
    out = []
    for i in range(0, trigs.shape[0], chunk):
        tb = trigs[i:i+chunk].to(dev); b = tb.shape[0]
        seq = torch.cat([pre.expand(b, -1), tb, suf.expand(b, -1), ans.expand(b, -1)], 1)
        lg = model(seq, **{_LTK: len(ans) + 1}).logits[:, :-1].float()
        out.append((lg.softmax(-1) @ cos).mean(-1))
        del lg, seq
    return torch.cat(out)

def grad_metric(trig, PRE_, SUF_, ans):
    """d(metric)/d(one-hot) — phase 6's proposer. [k, V]"""
    E  = model.model.embed_tokens.weight
    oh = F.one_hot(trig.to(dev), V).to(E.dtype).requires_grad_(True)
    inp = torch.cat([E[torch.tensor(PRE_, device=dev)], oh @ E,
                     E[torch.tensor(SUF_, device=dev)], E[ans]]).unsqueeze(0)
    lg = model(inputs_embeds=inp, **{_LTK: len(ans) + 1}).logits[0, :-1].float()
    (g,) = torch.autograd.grad((lg.softmax(-1) @ OBJ_COS).mean(), oh)
    del oh, inp, lg
    torch.cuda.empty_cache()
    return g.detach()

# ---------- the twist lens ----------------------------------------------------
class _Stop(Exception): pass

def _capture(store):
    def hook(mod, args, kwargs):
        store.append(args[0] if args else kwargs["hidden_states"])
        raise _Stop
    return hook

def h_at(L, *, input_ids=None, inputs_embeds=None):
    """Input to LAYERS[L], aborting the forward pass there. [B, T, d]"""
    assert 0 <= L < N_L, f"lens layer must be < {N_L}"
    store = []
    hd = LAYERS[L].register_forward_pre_hook(_capture(store), with_kwargs=True)
    try:
        if inputs_embeds is not None: model(inputs_embeds=inputs_embeds, use_cache=False)
        else:                         model(input_ids=input_ids, use_cache=False)
    except _Stop:
        pass
    finally:
        hd.remove()
    assert store, "hook never fired"
    return store[0]

@torch.no_grad()
def lens_batch(trigs, PRE_, SUF_, L, d_hat, chunk=16):
    """<h_L[last prompt position], d_hat> per trigger. [B]"""
    pre = torch.tensor(PRE_, device=dev); suf = torch.tensor(SUF_, device=dev)
    out = []
    for i in range(0, trigs.shape[0], chunk):
        tb = trigs[i:i+chunk].to(dev); b = tb.shape[0]
        seq = torch.cat([pre.expand(b, -1), tb, suf.expand(b, -1)], 1)
        out.append(h_at(L, input_ids=seq)[:, -1].float() @ d_hat)
        del seq
    return torch.cat(out)

def grad_lens(trig, PRE_, SUF_, L, d_hat):
    """d(<h_L[last], d_hat>)/d(one-hot) — the twist lens. [k, V]"""
    E  = model.model.embed_tokens.weight
    oh = F.one_hot(trig.to(dev), V).to(E.dtype).requires_grad_(True)
    inp = torch.cat([E[torch.tensor(PRE_, device=dev)], oh @ E,
                     E[torch.tensor(SUF_, device=dev)]]).unsqueeze(0)
    h = h_at(L, inputs_embeds=inp)[0, -1].float()
    (g,) = torch.autograd.grad(h @ d_hat, oh)
    del oh, inp, h
    torch.cuda.empty_cache()
    return g.detach()

# ---------- behavioural readout ----------------------------------------------
# Phase 6 §3: the steering vector installs the METAPHOR ("a bridge between the tangible
# and the abstract") while a genuine bridge question answers with the OBJECT — rivers,
# valleys, spans. That contrast is a free behavioural discriminator and is not fooled by
# lexical density the way the metric is.
CONCRETE = ["river", "valley", "road", "span", "arch", "cable", "deck", "steel",
            "concrete", "gorge", "railway", "truss", "pier", "suspension", "canyon",
            "creek", "highway", "beam", "pillar"]
METAPHOR = ["connect", "gap", "between", "understanding", "past", "future", "tangible",
            "abstract", "divide", "communities", "people", "worlds", "metaphor"]

def word_counts(text):
    t = text.lower()
    return sum(t.count(w) for w in CONCRETE), sum(t.count(w) for w in METAPHOR)

@torch.no_grad()
def readout(trig, PRE_, SUF_, tag="", n_samp=3, seed=0, verbose=True, lens_spec=None):
    """Greedy 160 + T=0.8/45 x n: four spaces, distinctness, concrete vs metaphor."""
    trig = torch.as_tensor(_tl(trig))
    ids  = torch.tensor([PRE_ + trig.tolist() + SUF_], device=dev)
    n_p  = ids.shape[1]
    gen  = model.generate(ids, max_new_tokens=160, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)[0]
    sc, H = score_answer(gen, n_p)
    ansg  = tokenizer.decode(gen[n_p:], skip_special_tokens=True)
    conc, meta = word_counts(ansg)
    r = dict(tag=tag, greedy=sc, greedy_H=H, greedy_distinct=distinct_ratio(gen[n_p:]),
             greedy_answer=ansg, concrete=conc, metaphor=meta,
             trigger=trig.tolist(), trigger_str=tokenizer.decode(trig.tolist()))
    ss, dd, texts = [], [], []
    for i in range(n_samp):
        torch.manual_seed(seed + i)
        g = model.generate(ids, max_new_tokens=45, do_sample=True, temperature=0.8,
                           top_p=0.95, pad_token_id=tokenizer.eos_token_id)[0]
        s, _ = score_answer(g, n_p)
        ss.append(s); dd.append(distinct_ratio(g[n_p:]))
        texts.append(tokenizer.decode(g[n_p:], skip_special_tokens=True))
    r["sampled"] = {k: sum(s[k] for s in ss) / len(ss) for k in KEYS}
    r["sampled_distinct"] = sum(dd) / len(dd)
    r["sampled_answers"]  = texts
    if lens_spec is not None:
        r["lens"] = lens_batch(trig.unsqueeze(0), PRE_, SUF_, *lens_spec)[0].item()
    if verbose:
        print(f"{tag:<38} greedy {r['greedy'][OBJ_KEY]:+.4f} | sampled "
              f"{r['sampled'][OBJ_KEY]:+.4f} | dist {r['greedy_distinct']:.2f} | "
              f"conc {conc:>2} meta {meta:>2}"
              + (f" | lens {r['lens']:+.2f}" if lens_spec is not None else ""))
        print(f"    {ansg[:150]!r}")
    torch.cuda.empty_cache()
    return r

print("machinery ready")

vocab 151936 -> usable 148023 | blocked 375 by substring + neighbours -> 659 (0.43%)   (phase 6: 148023 usable, 659 blocked)
machinery ready


## §3 — NEXT-STEPS item 3: what do the *working* phrases score?

Phase 6's RECIPE stage 6 requires this and it was never run: *"score the hand-written
interventions that work on the same objective … that comparison is cheap and it is the one
that settles whether an objective is worth optimising."* Phase 5 §5.1 measured these five in
an 8-slot channel — ` the user really loves bridges` reached **100%** on-topic, ` bridge`
alone 48.6%.

Two floors are included deliberately: a comma run (phase 7 §3b scores `','` at 0.0990,
*above* `tell me about bridges`) and a random junk trigger.

In [10]:
# === §3 — the working phrases under phase 6's metric ===
import torch

INTERVENTIONS = [
    ("phrase 100%  ' the user really loves bridges'", " the user really loves bridges"),
    ("phrase 80.6% ' bridges, obviously'",            " bridges, obviously"),
    ("phrase 77.8% ' I should mention bridges'",      " I should mention bridges"),
    ("phrase 66.7% ' that bridges are the answer'",   " that bridges are the answer"),
    ("phrase 48.6% ' bridge'",                        " bridge"),
    ("control      ' the user really loves puzzles'", " the user really loves puzzles"),
    ("floor        commas x 8",                       ",,,,,,,,"),
    ("floor        ' the' x 8",                       " the the the the the the the the"),
    ("floor        random 8 junk tokens",             None),
]
_g = torch.Generator().manual_seed(1)
PHRASE_RES = {}
for tag, txt in INTERVENTIONS:
    ids = (PIDX[torch.randint(len(PIDX), (8,), generator=_g)].tolist() if txt is None
           else tokenizer(txt, add_special_tokens=False).input_ids)
    PHRASE_RES[tag] = readout(ids, PRE, SUF, tag=tag, n_samp=3, seed=0)

print("\n" + "=" * 112)
print("Four-space table, greedy 160 — the phase 6 §2 / §6 comparison")
print(f"{'intervention':<40} " + " ".join(f"{k:>9}" for k in KEYS) + f" {'dist':>6}")
print("-" * 112)
for tag, r in PHRASE_RES.items():
    print(f"{tag:<40} " + " ".join(f"{r['greedy'][k]:>9.4f}" for k in KEYS)
          + f" {r['greedy_distinct']:>6.2f}")
print("-" * 112)
for label, qs in [("[control queries]", QUERIES[:4]), ("[bridge queries]", QUERIES[4:])]:
    print(f"{label:<40} " + " ".join(
        f"{BAND(k, qs)[0]:>4.3f}/{BAND(k, qs)[1]:<4.3f}" for k in KEYS))
print(f"{'[phase 6 GCG winner, out.cent]':<40} 0.0620")

phrase 100%  ' the user really loves bridges' greedy +0.0896 | sampled +0.0761 | dist 0.68 | conc  6 meta  1
    "That's a great question! Since the user really loves bridges, I can suggest a variety of fun and meaningful activities that celebrate bridges — both l"
phrase 80.6% ' bridges, obviously'     greedy +0.0984 | sampled +0.0663 | dist 0.67 | conc  0 meta  4
    'Ah, "Bridges, obviously" — that\'s a clever and poetic way to start your day! It\'s like you\'re saying, "I\'m going to build something meaningful today, '
phrase 77.8% ' I should mention bridges' greedy +0.1074 | sampled +0.0740 | dist 0.66 | conc  1 meta  4
    "That's a great idea! Bridges are fascinating structures that connect places, ideas, and even people. Here's a fun and creative list of things you can "
phrase 66.7% ' that bridges are the answer' greedy +0.0870 | sampled +0.0764 | dist 0.64 | conc  0 meta  6
    "It sounds like you're looking for a creative or meaningful activity that connects to the idea of *

## §4 — phase 2 §4's ladder: at what depth should the lens read?

Phase 2's open thread #2: *"optimise the lens at layer 32, not 18 — the alignment signal
peaks there."* It was measured with a **degradation ladder**: revert *j* of a working
intervention's slots to random pool tokens, which spans behaviour from ceiling to floor
*within one target*, then correlate behaviour against alignment at each layer.

Alignment is the **cue-minus-neutral steering delta**, never the raw state:
`cos(h_L(trigger) − h_L(neutral), d)`. Raw hidden-state cosine between two different
triggers in the same scaffold is 0.9914 before any search (phase 7 §4) — a context floor,
not a signal. Correlations are reported both raw and residualised on *j*, because *j* drives
both sides.

In [11]:
# === §4 — degradation ladder -> the lens depth ===
import torch, torch.nn.functional as F, math

PH_IDS   = tokenizer(PHRASE, add_special_tokens=False).input_ids
NEUT_IDS = tokenizer(" the user really loves puzzles", add_special_tokens=False).input_ids
K_PH     = len(PH_IDS)
REPS     = 3
H_NEUT   = all_h_last(PRE + NEUT_IDS + SUF)

rows = []
_g = torch.Generator().manual_seed(0)
for j in range(K_PH + 1):
    for rep in range(REPS if 0 < j < K_PH else 1):
        ids = list(PH_IDS)
        for s in torch.randperm(K_PH, generator=_g)[:j].tolist():
            ids[s] = int(PIDX[torch.randint(len(PIDX), (1,), generator=_g)])
        full = PRE + ids + SUF
        gen  = model.generate(torch.tensor([full], device=dev), max_new_tokens=45,
                              do_sample=False, pad_token_id=tokenizer.eos_token_id)[0]
        sc, _ = score_answer(gen, len(full))
        H = all_h_last(full)
        rows.append(dict(
            j=j, rep=rep, score=sc[OBJ_KEY], scores=sc,
            dcos_ph =F.cosine_similarity(H - H_NEUT, D_PH,  dim=-1).tolist(),
            dcos_caa=F.cosine_similarity(H - H_NEUT, V_CAA, dim=-1).tolist(),
            proj_ph =(H * DP_HAT).sum(-1).tolist(),
            distinct=distinct_ratio(gen[len(full):]),
            answer=tokenizer.decode(gen[len(full):], skip_special_tokens=True)[:120]))
        print(f"  j={j} rep={rep}  {OBJ_KEY}={sc[OBJ_KEY]:.4f}  "
              f"cos@L16={rows[-1]['dcos_ph'][16]:+.3f}  {rows[-1]['answer'][:58]!r}")
        torch.cuda.empty_cache()

def pearson(a, b):
    n = len(a); ma, mb = sum(a)/n, sum(b)/n
    va = sum((x-ma)**2 for x in a); vb = sum((y-mb)**2 for y in b)
    if va == 0 or vb == 0: return float("nan")
    return sum((x-ma)*(y-mb) for x, y in zip(a, b)) / math.sqrt(va*vb)

def resid(v, j):                       # residualise on the degradation level
    m = {}
    for jj, x in zip(j, v): m.setdefault(jj, []).append(x)
    m = {k: sum(x)/len(x) for k, x in m.items()}
    return [x - m[jj] for jj, x in zip(j, v)]

S = [r["score"] for r in rows]; J = [r["j"] for r in rows]
print(f"\n{'L':>3} {'r(score,cos D_PH)':>18} {'partial|j':>10} "
      f"{'r(score,cos V_CAA)':>19} {'partial|j':>10}")
LADDER_CORR = {}
for L in range(0, N_L + 1, 2):
    a = [r["dcos_ph"][L] for r in rows]; b = [r["dcos_caa"][L] for r in rows]
    LADDER_CORR[L] = dict(r_ph=pearson(S, a), p_ph=pearson(resid(S, J), resid(a, J)),
                          r_caa=pearson(S, b), p_caa=pearson(resid(S, J), resid(b, J)))
    c = LADDER_CORR[L]
    print(f"{L:>3} {c['r_ph']:>18.3f} {c['p_ph']:>10.3f} {c['r_caa']:>19.3f} "
          f"{c['p_caa']:>10.3f}")

_ok = [L for L, c in LADDER_CORR.items() if L < N_L and c["p_ph"] == c["p_ph"]]
L_LENS = max(_ok, key=lambda L: LADDER_CORR[L]["p_ph"])
D_HAT  = DP_HAT[L_LENS].contiguous()
print(f"\nlens depth chosen: L{L_LENS} — highest partial r for D_PH "
      f"({LADDER_CORR[L_LENS]['p_ph']:+.3f}), residualised on j")
print("phase 2 ran its lens at L18 of 36 and found the peak at L32 (+0.86 there, +0.70 at "
      "L18); this is that measurement on the bridge problem.")

  j=0 rep=0  out.cent=0.0745  cos@L16=+0.551  "That's a great question! Since the user really loves bridg"
  j=1 rep=0  out.cent=0.0652  cos@L16=+0.101  'It looks like your message is a bit unclear or possibly co'
  j=1 rep=1  out.cent=0.0643  cos@L16=+0.288  'It seems like your message might be a bit unclear or misty'
  j=1 rep=2  out.cent=0.0657  cos@L16=+0.179  "It sounds like you're interested in exploring the intersec"
  j=2 rep=0  out.cent=0.0642  cos@L16=+0.067  "It sounds like you're asking for a fun or creative activit"
  j=2 rep=1  out.cent=0.0659  cos@L16=+0.242  "It sounds like you're in a playful, creative mood! The phr"
  j=2 rep=2  out.cent=0.0501  cos@L16=+0.079  "It sounds like you're looking for some fun or meaningful a"
  j=3 rep=0  out.cent=0.0488  cos@L16=-0.035  'It looks like you\'re asking, *"What shall I do today?"* an'
  j=3 rep=1  out.cent=0.0643  cos@L16=+0.039  'It looks like your message is a bit unclear. Let me try to'
  j=3 rep=2  out.cent=0.0468  cos@L1

## §5 — three proposers, one accept test, equal budget

The only thing that differs between the arms is **what proposes candidate substitutions**.
Every arm accepts on the same test — phase 6's metric, verified by a real forward pass on the
current greedy rollout — with the same `k=53`, `n_mut=7`, `n_cand=256`, `n_top=512`, full
pool, suffix position, repeat init, seed 1, and the same wall-clock budget. That is phase 6's
winning configuration (trial 12, `true = 0.0620`).

- **`metric`** — phase 6's own gradient.
- **`lens`** — the twist lens at the depth §4 chose.
- **`random`** — no gradient at all, candidates drawn uniformly from the pool. This is
  **NEXT-STEPS item 1**, which the project has never had: an equal-compute random arm. If it
  matches the gradient arms, "GCG maximised the objective" means "random search nudged it".

`pred_corr` is measured exactly as in phase 2 — the gradient's predicted improvement against
the improvement the forward pass actually delivered, over every candidate at every step.

In [12]:
# === §5 — the search, with a pluggable proposer and accept test ===
import torch, time

def gcg3(proposer="metric", accept="metric", lens_L=None, d_hat=None,
         q=Q, k=53, position="suffix", pool_kind="full", n_top=512, n_cand=256,
         n_mut=7, n_new=48, refresh_every=2, chunk=16, budget_s=240, seed=1,
         init="repeat", log_every=8):
    """Phase 6 trial 12's configuration; `proposer` and `accept` are the only knobs."""
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    PRE_, SUF_ = make_scaffold(q, position)
    pool   = build_pool(pool_kind); pool_d = pool.to(dev)
    pidx   = torch.nonzero(pool).squeeze(-1)
    trig   = (pidx[torch.randint(len(pidx), (1,), generator=gen)].repeat(k) if init == "repeat"
              else pidx[torch.randint(len(pidx), (k,), generator=gen)])
    needs_rollout = (accept == "metric") or (proposer == "metric")
    ans = rollout(trig, PRE_, SUF_, n_new) if needs_rollout else None

    def acc(tt):
        return (score_batch(tt, PRE_, SUF_, ans, chunk=chunk) if accept == "metric"
                else lens_batch(tt, PRE_, SUF_, lens_L, d_hat, chunk=chunk))

    cur = acc(trig.unsqueeze(0))[0].item()
    best, best_t = cur, trig.clone()
    hist, preds, reals = [], [], []
    ar = torch.arange(k, device=dev)
    t0, s, n_evals = time.time(), 0, 0

    while time.time() - t0 < budget_s:
        if proposer == "random":
            g = None
            pool_picks = pidx[torch.randint(len(pidx), (n_mut, n_cand), generator=gen)].to(dev)
        else:
            g = (grad_metric(trig, PRE_, SUF_, ans) if proposer == "metric"
                 else grad_lens(trig, PRE_, SUF_, lens_L, d_hat))
            g[:, ~pool_d] = -float("inf")
            top = g.topk(min(n_top, int(pool.sum())), dim=-1).indices
        cands = trig.unsqueeze(0).repeat(n_cand, 1).to(dev)
        for m in range(n_mut):
            slots = torch.randint(k, (n_cand,), generator=gen)
            picks = (pool_picks[m] if g is None
                     else top[slots, torch.randint(top.shape[1], (n_cand,), generator=gen)])
            cands[torch.arange(n_cand), slots] = picks
        sc = acc(cands); n_evals += n_cand

        if g is not None:                       # phase 2's pred_corr, per candidate
            old   = trig.to(dev)
            g_new = g[ar.unsqueeze(0).expand(n_cand, k), cands].float()     # [n_cand, k]
            g_old = g[ar, old].float().unsqueeze(0)                          # [1, k]
            pred  = ((g_new - g_old) * (cands != old.unsqueeze(0))).sum(-1)
            preds.append(pred.cpu()); reals.append((sc - cur).float().cpu())
            del g_new, g_old, pred, top

        j = int(sc.argmax())
        trig, cur = cands[j].cpu(), sc[j].item()
        s += 1
        if s % refresh_every == 0:
            if needs_rollout: ans = rollout(trig, PRE_, SUF_, n_new)
            cur = acc(trig.unsqueeze(0))[0].item()
            if cur > best: best, best_t = cur, trig.clone()
            hist.append((s, cur, best, round(time.time() - t0, 1)))
            if log_every and s % log_every == 0:
                print(f"    step {s:>3}  {accept}={cur:.4f}  best={best:.4f}  "
                      f"{time.time()-t0:.0f}s")
        del cands, sc
        if g is not None: del g
        torch.cuda.empty_cache()

    pc = float("nan")
    if preds:
        pr, rl = torch.cat(preds), torch.cat(reals)
        m = torch.isfinite(pr) & torch.isfinite(rl)
        if int(m.sum()) > 2:
            pc = float(torch.corrcoef(torch.stack([pr[m], rl[m]]))[0, 1])
    return dict(trigger=best_t, best_accept=best, steps=s, n_evals=n_evals, pred_corr=pc,
                hist=hist, proposer=proposer, accept=accept, secs=time.time() - t0,
                PRE=PRE_, SUF=SUF_)

print("gcg3 ready")

gcg3 ready


In [13]:
# === §5 — run the three arms ===
import torch

BUDGET = 240
ARMS = {}
for proposer in ["metric", "lens", "random"]:
    print(f"\n=== proposer = {proposer} | accept = metric | {BUDGET}s ===")
    r = gcg3(proposer=proposer, accept="metric", lens_L=L_LENS, d_hat=D_HAT,
             budget_s=BUDGET, seed=1)
    print(f"  steps {r['steps']}  evals {r['n_evals']}  best(teacher-forced) "
          f"{r['best_accept']:.4f}  pred_corr {r['pred_corr']:+.3f}")
    r["readout"] = readout(r["trigger"], r["PRE"], r["SUF"],
                           tag=f"{proposer}-proposer", lens_spec=(L_LENS, D_HAT))
    ARMS[proposer] = r
    torch.cuda.empty_cache()

print("\n" + "=" * 108)
print(f"{'proposer':<10} {'steps':>6} {'evals':>7} {'pred_corr':>10} {'tf best':>9} "
      f"{'greedy':>9} {'sampled':>9} {'dist':>6} {'lens':>9} {'conc/meta':>10}")
print("-" * 108)
for p, r in ARMS.items():
    ro = r["readout"]
    print(f"{p:<10} {r['steps']:>6} {r['n_evals']:>7} {r['pred_corr']:>10.3f} "
          f"{r['best_accept']:>9.4f} {ro['greedy'][OBJ_KEY]:>9.4f} "
          f"{ro['sampled'][OBJ_KEY]:>9.4f} {ro['greedy_distinct']:>6.2f} "
          f"{ro['lens']:>9.2f} {str(ro['concrete'])+'/'+str(ro['metaphor']):>10}")
print("-" * 108)
print(f"phase 2 pred_corr: logit −0.192, lens +0.348 (8 animals, 4B) | phase 6 GCG winner "
      f"0.0620 | control band {CONTROL_BAND[0]:.4f}–{CONTROL_BAND[1]:.4f}")


=== proposer = metric | accept = metric | 240s ===
    step   8  metric=0.0520  best=0.0520  35s
    step  16  metric=0.0604  best=0.0604  70s
    step  24  metric=0.0575  best=0.0604  104s
    step  32  metric=0.0577  best=0.0604  139s
    step  40  metric=0.0575  best=0.0623  174s
    step  48  metric=0.0590  best=0.0623  209s
    step  56  metric=0.0575  best=0.0623  244s
  steps 56  evals 14336  best(teacher-forced) 0.0623  pred_corr -0.106
metric-proposer                        greedy +0.0481 | sampled +0.0593 | dist 0.74 | conc  0 meta  0 | lens +0.08
    "It looks like you've shared a mix of random characters, words, and phrases that don't form a coherent message. It might be a puzzle, a test, or just a"

=== proposer = lens | accept = metric | 240s ===
    step   8  metric=0.0565  best=0.0565  34s
    step  16  metric=0.0510  best=0.0565  67s
    step  24  metric=0.0595  best=0.0595  101s
    step  32  metric=0.0541  best=0.0601  135s
    step  40  metric=0.0561  best=0.0601  

## §6 — the lens as the objective itself

§5 uses the lens only to *propose*; the accept test is still the metric. This section
optimises the projection directly — phase 7's untested *projection* objective, pointed at the
direction phase 5's 100% intervention actually moves rather than at the metric's own gradient.
It has no threshold (unlike phase 7 §5's L2 target, which needed `cos > 0.329` where token
moves deliver 0.02), so every positive-cosine move contributes.

The readout is what decides it: four spaces, distinctness, concrete-vs-metaphor, and the
projection the working phrase itself reaches. A lens-maximising trigger that beats the working
phrase's projection while producing none of its behaviour makes the twist lens the sixth
objective to fall; an ordering that holds makes the direction worth searching in.

In [14]:
# === §6 — optimise the lens projection directly, then read out ===
import torch

print(f"=== proposer = lens | accept = lens (L{L_LENS}) | {BUDGET}s ===")
LENS_ARM = gcg3(proposer="lens", accept="lens", lens_L=L_LENS, d_hat=D_HAT,
                budget_s=BUDGET, seed=1)
print(f"  steps {LENS_ARM['steps']}  evals {LENS_ARM['n_evals']}  "
      f"best projection {LENS_ARM['best_accept']:.3f}  pred_corr {LENS_ARM['pred_corr']:+.3f}")
LENS_ARM["readout"] = readout(LENS_ARM["trigger"], LENS_ARM["PRE"], LENS_ARM["SUF"],
                              tag="lens-objective", lens_spec=(L_LENS, D_HAT))

REFS = {}
for tag, txt in [("' the user really loves bridges'", " the user really loves bridges"),
                 ("' bridge'", " bridge"),
                 ("' the user really loves puzzles'", " the user really loves puzzles"),
                 ("commas x 8", ",,,,,,,,")]:
    ids = tokenizer(txt, add_special_tokens=False).input_ids
    REFS[tag] = lens_batch(torch.tensor([ids]), PRE, SUF, L_LENS, D_HAT)[0].item()
_g2 = torch.Generator().manual_seed(7)
REFS["random 53 junk tokens"] = lens_batch(
    PIDX[torch.randint(len(PIDX), (1, 53), generator=_g2)], PRE, SUF, L_LENS, D_HAT)[0].item()
REFS["lens-optimised trigger"] = LENS_ARM["readout"]["lens"]

print(f"\nlens projection <h_L{L_LENS}[last], d_hat>:")
for tag, v in sorted(REFS.items(), key=lambda kv: -kv[1]):
    mark = "  <-- optimised" if tag.startswith("lens-opt") else ""
    print(f"  {tag:<36} {v:+8.2f}{mark}")

print(f"\nfour spaces, greedy 160:")
print(f"{'':<32} " + " ".join(f"{k:>9}" for k in KEYS) + f" {'dist':>6} {'c/m':>6}")
TABLE = [("lens-objective trigger",  LENS_ARM["readout"]),
         ("lens-proposer trigger",   ARMS["lens"]["readout"]),
         ("metric-proposer trigger", ARMS["metric"]["readout"]),
         ("random-proposer trigger", ARMS["random"]["readout"]),
         ("' the user really loves bridges'",
          PHRASE_RES["phrase 100%  ' the user really loves bridges'"])]
for name, ro in TABLE:
    print(f"{name:<32} " + " ".join(f"{ro['greedy'][k]:>9.4f}" for k in KEYS)
          + f" {ro['greedy_distinct']:>6.2f} "
          + f"{str(ro['concrete'])+'/'+str(ro['metaphor']):>6}")
for label, qs in [("[control queries]", QUERIES[:4]), ("[bridge queries]", QUERIES[4:])]:
    print(f"{label:<32} " + " ".join(
        f"{BAND(k, qs)[0]:>4.3f}/{BAND(k, qs)[1]:<4.3f}" for k in KEYS))

print("\ntriggers:")
for name, ro in TABLE[:4]:
    print(f"\n  {name}: {ro['trigger_str'][:180]!r}")
    print(f"    -> {ro['greedy_answer'][:200]!r}")

=== proposer = lens | accept = lens (L8) | 240s ===
    step   8  lens=3.5791  best=3.5791  3s
    step  16  lens=4.6173  best=4.6173  6s
    step  24  lens=5.2078  best=5.2078  9s
    step  32  lens=5.3549  best=5.3889  12s
    step  40  lens=5.2844  best=5.4671  16s
    step  48  lens=5.5876  best=5.5876  19s
    step  56  lens=5.6181  best=5.6511  22s
    step  64  lens=5.5621  best=5.6511  25s
    step  72  lens=5.6794  best=5.7333  28s
    step  80  lens=5.3593  best=5.7333  31s
    step  88  lens=5.2120  best=5.7333  34s
    step  96  lens=5.2845  best=5.7333  37s
    step 104  lens=5.5292  best=5.7333  40s
    step 112  lens=5.7655  best=5.7655  44s
    step 120  lens=5.4066  best=5.7655  47s
    step 128  lens=5.6324  best=5.7655  50s
    step 136  lens=5.6120  best=5.7655  53s
    step 144  lens=5.6004  best=5.7655  56s
    step 152  lens=5.4753  best=5.7655  59s
    step 160  lens=5.4712  best=5.7655  62s
    step 168  lens=5.5771  best=5.7655  65s
    step 176  lens=5.6681  

In [15]:
# === §6.1 — norm or direction? decomposing the projection ===
# <h, d_hat> = ||h|| * cos(h, d_hat), and nothing in the objective bounds ||h||. Phase 7 §3b's
# lesson is that an unbounded objective gets gamed by whatever is cheapest, so decompose it:
#   ||h||                    how big the state got
#   cos(h - h_ref, d_hat)    the aligned part of the DISPLACEMENT (phase 5's context floor
#                            means raw cos(h, d_hat) is not the quantity of interest)
# h_ref is the run's own random init trigger, so every row is measured against the same origin.
import torch, torch.nn.functional as F

_g3 = torch.Generator().manual_seed(1)
_pool = build_pool("full"); _pi = torch.nonzero(_pool).squeeze(-1)
REF_TRIG = _pi[torch.randint(len(_pi), (1,), generator=_g3)].repeat(53)   # gcg3's own init

@torch.no_grad()
def h_last(ids):
    seq = torch.tensor([PRE + _tl(ids) + SUF], device=dev)
    return h_at(L_LENS, input_ids=seq)[0, -1].float()

H_REF = h_last(REF_TRIG)
ROWS = [
    ("lens-objective trigger",           LENS_ARM["readout"]["trigger"]),
    ("lens-proposer trigger",            ARMS["lens"]["readout"]["trigger"]),
    ("metric-proposer trigger",          ARMS["metric"]["readout"]["trigger"]),
    ("random-proposer trigger",          ARMS["random"]["readout"]["trigger"]),
    ("' the user really loves bridges'", tokenizer(PHRASE, add_special_tokens=False).input_ids),
    ("' bridge'",                        tokenizer(" bridge", add_special_tokens=False).input_ids),
    ("' the user really loves puzzles'", tokenizer(" the user really loves puzzles",
                                                   add_special_tokens=False).input_ids),
    ("commas x 8",                       tokenizer(",,,,,,,,", add_special_tokens=False).input_ids),
    ("random 53 (gcg3 init)",            REF_TRIG),
]
DECOMP = {}
print(f"{'':<34} {'<h,d>':>8} {'||h||':>8} {'cos(h,d)':>9} {'||h-ref||':>10} {'cos(h-ref,d)':>13}")
print("-" * 90)
for name, ids in ROWS:
    h = h_last(ids)
    d = h - H_REF
    DECOMP[name] = dict(proj=float(h @ D_HAT), norm=float(h.norm()),
                        cos_raw=float(F.cosine_similarity(h, D_HAT, dim=0)),
                        delta_norm=float(d.norm()),
                        cos_delta=float(F.cosine_similarity(d, D_HAT, dim=0)))
    r = DECOMP[name]
    print(f"{name:<34} {r['proj']:>8.2f} {r['norm']:>8.2f} {r['cos_raw']:>9.4f} "
          f"{r['delta_norm']:>10.2f} {r['cos_delta']:>13.4f}")
print("-" * 90)
print(f"mean non-sink ||h|| at L{L_LENS}: {NONSINK[L_LENS]:.2f}   |   ||D_PH[L{L_LENS}]|| = "
      f"{D_PH[L_LENS].norm():.2f}")

                                      <h,d>    ||h||  cos(h,d)  ||h-ref||  cos(h-ref,d)
------------------------------------------------------------------------------------------
lens-objective trigger                 6.40    41.16    0.1554      17.78        0.3711
lens-proposer trigger                  0.80    38.99    0.0205       9.77        0.1024
metric-proposer trigger                0.08    38.21    0.0021       9.34        0.0303
random-proposer trigger                0.07    39.26    0.0017       9.50        0.0281
' the user really loves bridges'       2.30    38.52    0.0597       9.21        0.2717
' bridge'                              1.86    38.09    0.0488       8.37        0.2465
' the user really loves puzzles'       1.24    38.58    0.0321       9.09        0.1585
commas x 8                             1.38    37.90    0.0364       8.48        0.1865
random 53 (gcg3 init)                 -0.20    38.21   -0.0053       0.00        0.0000
-----------------------------

In [16]:
# === §6.2 — is that alignment bridge-specific, or just "a fluent phrase is in the slots"? ===
# §6.1 shows commas (0.187) and the puzzles control (0.159) already align with d_hat, measured
# from the junk init. So part of d_hat's apparent signal is "the slots contain ordinary text",
# not "the slots are about bridges" — and the §4 ladder cannot separate the two, because
# reverting slots to junk destroys both at once.
#
# Split it. F_DIR = mean displacement of the 8 matched CONTROL phrases from the junk init
# ("fluent phrase in the slots"); D_PERP = the part of d_hat orthogonal to F_DIR (bridge-specific).
import torch, torch.nn.functional as F

CTRL_H = torch.stack([h_last(tokenizer(p, add_special_tokens=False).input_ids) - H_REF
                      for p in PH_NEGS])
F_DIR  = F.normalize(CTRL_H.mean(0), dim=0)
D_PERP = F.normalize(D_HAT - (D_HAT @ F_DIR) * F_DIR, dim=0)
print(f"cos(d_hat, F_DIR) = {float(D_HAT @ F_DIR):+.3f}   "
      f"-> {100*float(D_HAT @ F_DIR)**2:.1f}% of the lens direction is the fluency component")

print(f"\n{'':<34} {'cos(.,d_hat)':>12} {'cos(.,F_DIR)':>12} {'cos(.,D_PERP)':>13}")
print("-" * 74)
SPLIT = {}
for name, ids in ROWS:
    d = h_last(ids) - H_REF
    SPLIT[name] = dict(d_hat=float(F.cosine_similarity(d, D_HAT, dim=0)),
                       fluency=float(F.cosine_similarity(d, F_DIR, dim=0)),
                       bridge=float(F.cosine_similarity(d, D_PERP, dim=0)))
    s = SPLIT[name]
    print(f"{name:<34} {s['d_hat']:>12.4f} {s['fluency']:>12.4f} {s['bridge']:>13.4f}")
print("-" * 74)
print("D_PERP is the bridge-specific axis: the working phrase should lead on it if the\n"
      "direction means what §4's r=+0.98 suggests, and the lens trigger's advantage should\n"
      "sit on F_DIR if the search found the cheap component instead.")

cos(d_hat, F_DIR) = +0.125   -> 1.6% of the lens direction is the fluency component

                                   cos(.,d_hat) cos(.,F_DIR) cos(.,D_PERP)
--------------------------------------------------------------------------
lens-objective trigger                   0.3711       0.3836        0.3256
lens-proposer trigger                    0.1024       0.5171        0.0378
metric-proposer trigger                  0.0303       0.4883       -0.0312
random-proposer trigger                  0.0281       0.4041       -0.0228
' the user really loves bridges'         0.2717       0.9889        0.1489
' bridge'                                0.2465       0.8892        0.1360
' the user really loves puzzles'         0.1585       0.9756        0.0364
commas x 8                               0.1865       0.8466        0.0809
random 53 (gcg3 init)                    0.0000       0.0000        0.0000
--------------------------------------------------------------------------
D_PERP is the b

In [17]:
# === Save everything ===
import json, torch

def _clean(r):
    d = {k: v for k, v in r.items() if k not in ("PRE", "SUF")}
    if torch.is_tensor(d.get("trigger")):
        d["trigger"] = d["trigger"].tolist()
        d["trigger_str"] = tokenizer(d["trigger"]) if False else tokenizer.decode(d["trigger"])
    if "readout" in d: d["readout"] = _clean(d["readout"])
    return d

OUT = dict(
    meta=dict(model=MODEL_ID, dtype=str(DTYPE), gpu=torch.cuda.get_device_name(0),
              query=Q, k=53, obj_key=OBJ_KEY, lens_layer=int(L_LENS), budget_s=BUDGET,
              phrase=PHRASE, n_negatives=len(PH_NEGS), transformers=transformers.__version__),
    rig=RIG,
    bands=dict(control={k: BAND(k, QUERIES[:4]) for k in KEYS},
               bridge={k: BAND(k, QUERIES[4:]) for k in KEYS},
               uniform={k: COS[k].mean().item() for k in KEYS}),
    directions=dict(
        cos_V_D=[float(torch.nn.functional.cosine_similarity(V_CAA[L], D_PH[L], dim=0))
                 for L in range(N_L + 1)],
        norm_V=[float(V_CAA[L].norm()) for L in range(N_L + 1)],
        norm_D=[float(D_PH[L].norm()) for L in range(N_L + 1)],
        nonsink_h=[float(x) for x in NONSINK]),
    phrases={k: _clean(v) for k, v in PHRASE_RES.items()},
    ladder=dict(rows=rows, corr={str(k): v for k, v in LADDER_CORR.items()},
                chosen_layer=int(L_LENS)),
    arms={k: _clean(v) for k, v in ARMS.items()},
    lens_arm=_clean(LENS_ARM),
    lens_refs=REFS,
    decomposition=DECOMP,
    fluency_split=dict(cos_dhat_fdir=float(D_HAT @ F_DIR), rows=SPLIT),
)
with open("phase8_twist_lens.json", "w") as f:
    json.dump(OUT, f, indent=1, ensure_ascii=False)
print("wrote phase8_twist_lens.json",
      f"({len(json.dumps(OUT, ensure_ascii=False))/1024:.0f} KB)")

wrote phase8_twist_lens.json (101 KB)


In [20]:
# === §7 — the confound guard: does the gradient beat random on a target GCG is good at? ===
#
# NEXT-STEPS item 2 asks whether GCG has ever succeeded on Qwen3-8B. Phase 4 says yes — but at
# NEXT-TOKEN control with a prefilled answer slot (0.9951 / 0.9991), which phase 5 §5.1 already
# flagged as a different claim from sustained behaviour. The version that actually guards §5's
# result is narrower and about THIS rig: with this pool, scaffold, budget and code, does the
# gradient beat random on an easy next-token objective? If yes, pred_corr ~ 0 and random == GCG
# are properties of the bridgeness objective. If no, they are properties of the implementation
# or the backbone, and §5 means much less.
#
# The literal target string is banned from the pool, so neither arm can simply write it.
import torch, torch.nn.functional as F, time, unicodedata

CAND_TARGETS = [" Sure", " wolf", " banana"]
TARGETS = []
for t in CAND_TARGETS:
    ids = tokenizer(t, add_special_tokens=False).input_ids
    if len(ids) == 1: TARGETS.append((t, ids[0]))
    else: print(f"skipping {t!r}: {len(ids)} tokens")
TARGETS = TARGETS[:2]
print("targets:", [(t, i) for t, i in TARGETS])

def ban_mask(word):
    w = unicodedata.normalize("NFKD", word.strip()).casefold()
    m = torch.zeros(V, dtype=torch.bool)
    for i, s in enumerate(NFKD):
        if s and w in s: m[i] = True
    return m

@torch.no_grad()
def p_target_batch(trigs, PRE_, SUF_, tid, chunk=32):
    pre = torch.tensor(PRE_, device=dev); suf = torch.tensor(SUF_, device=dev)
    out = []
    for i in range(0, trigs.shape[0], chunk):
        tb = trigs[i:i+chunk].to(dev); b = tb.shape[0]
        seq = torch.cat([pre.expand(b, -1), tb, suf.expand(b, -1)], 1)
        lg = model(seq, **{_LTK: 1}).logits[:, -1].float()
        out.append(lg.softmax(-1)[:, tid])
        del lg, seq
    return torch.cat(out)

def grad_target(trig, PRE_, SUF_, tid):
    E  = model.model.embed_tokens.weight
    oh = F.one_hot(trig.to(dev), V).to(E.dtype).requires_grad_(True)
    inp = torch.cat([E[torch.tensor(PRE_, device=dev)], oh @ E,
                     E[torch.tensor(SUF_, device=dev)]]).unsqueeze(0)
    lg = model(inputs_embeds=inp, **{_LTK: 1}).logits[0, -1].float()
    (g,) = torch.autograd.grad(lg.log_softmax(-1)[tid], oh)
    del oh, inp, lg
    torch.cuda.empty_cache()
    return g.detach()

def gcg_nexttok(tid, banned, proposer="grad", k=53, q=Q, position="suffix",
                n_top=512, n_cand=256, n_mut=7, chunk=32, budget_s=120, seed=1,
                init="repeat", log_every=25):
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    PRE_, SUF_ = make_scaffold(q, position)
    pool = build_pool("full") & ~banned
    pool_d = pool.to(dev); pidx = torch.nonzero(pool).squeeze(-1)
    trig = (pidx[torch.randint(len(pidx), (1,), generator=gen)].repeat(k) if init == "repeat"
            else pidx[torch.randint(len(pidx), (k,), generator=gen)])
    cur = p_target_batch(trig.unsqueeze(0), PRE_, SUF_, tid, chunk)[0].item()
    p0, best, best_t = cur, cur, trig.clone()
    hist, preds, reals = [(0, cur, 0.0)], [], []
    ar = torch.arange(k, device=dev)
    t0, s, n_evals = time.time(), 0, 0
    while time.time() - t0 < budget_s:
        if proposer == "random":
            g = None
            pool_picks = pidx[torch.randint(len(pidx), (n_mut, n_cand), generator=gen)].to(dev)
        else:
            g = grad_target(trig, PRE_, SUF_, tid)
            g[:, ~pool_d] = -float("inf")
            top = g.topk(min(n_top, int(pool.sum())), dim=-1).indices
        cands = trig.unsqueeze(0).repeat(n_cand, 1).to(dev)
        for m in range(n_mut):
            slots = torch.randint(k, (n_cand,), generator=gen)
            picks = (pool_picks[m] if g is None
                     else top[slots, torch.randint(top.shape[1], (n_cand,), generator=gen)])
            cands[torch.arange(n_cand), slots] = picks
        sc = p_target_batch(cands, PRE_, SUF_, tid, chunk); n_evals += n_cand
        if g is not None:
            old = trig.to(dev)
            gn = g[ar.unsqueeze(0).expand(n_cand, k), cands].float()
            go = g[ar, old].float().unsqueeze(0)
            preds.append((((gn - go) * (cands != old.unsqueeze(0))).sum(-1)).cpu())
            reals.append((sc - cur).float().cpu())
            del gn, go, top
        j = int(sc.argmax()); trig, cur = cands[j].cpu(), sc[j].item()
        if cur > best: best, best_t = cur, trig.clone()
        s += 1
        hist.append((s, cur, round(time.time() - t0, 1)))
        if log_every and s % log_every == 0:
            print(f"    step {s:>3}  p={cur:.4f}  best={best:.4f}  {time.time()-t0:.0f}s")
        del cands, sc
        if g is not None: del g
        torch.cuda.empty_cache()
    pc = float("nan")
    if preds:
        pr, rl = torch.cat(preds), torch.cat(reals)
        m = torch.isfinite(pr) & torch.isfinite(rl)
        if int(m.sum()) > 2: pc = float(torch.corrcoef(torch.stack([pr[m], rl[m]]))[0, 1])
    return dict(p0=p0, best=best, steps=s, n_evals=n_evals, pred_corr=pc, hist=hist,
                trigger=best_t, proposer=proposer)

GUARD = {}
for tstr, tid in TARGETS:
    banned = ban_mask(tstr)
    print(f"\n### target {tstr!r} (id {tid}) | {int(banned.sum())} tokens banned by substring")
    for proposer in ["grad", "random"]:
        print(f"  --- proposer = {proposer} ---")
        GUARD[(tstr, proposer)] = gcg_nexttok(tid, banned, proposer=proposer, budget_s=120)
        r = GUARD[(tstr, proposer)]
        print(f"  p {r['p0']:.4f} -> {r['best']:.4f} | steps {r['steps']} evals {r['n_evals']} "
              f"| pred_corr {r['pred_corr']:+.3f}")

print("\n" + "=" * 92)
print(f"{'target':<10} {'proposer':<9} {'p start':>9} {'p best':>9} {'x gain':>8} "
      f"{'steps':>6} {'evals':>7} {'pred_corr':>10}")
print("-" * 92)
for (tstr, prop), r in GUARD.items():
    print(f"{tstr:<10} {prop:<9} {r['p0']:>9.4f} {r['best']:>9.4f} "
          f"{r['best']/max(r['p0'],1e-9):>8.1f} {r['steps']:>6} {r['n_evals']:>7} "
          f"{r['pred_corr']:>10.3f}")
print("-" * 92)
print("§5 for comparison (bridgeness metric): metric-grad 0.0623 / random 0.0622, "
      "pred_corr -0.106 / --")

targets: [(' Sure', 22555), (' wolf', 36542)]

### target ' Sure' (id 22555) | 88 tokens banned by substring
  --- proposer = grad ---
    step  25  p=0.0002  best=0.0003  46s
    step  50  p=0.0012  best=0.0012  92s
  p 0.0000 -> 0.0019 | steps 65 evals 16640 | pred_corr -0.033
  --- proposer = random ---
    step  25  p=0.0001  best=0.0001  42s
    step  50  p=0.0002  best=0.0002  85s
  p 0.0000 -> 0.0007 | steps 71 evals 18176 | pred_corr +nan

### target ' wolf' (id 36542) | 8 tokens banned by substring
  --- proposer = grad ---
    step  25  p=0.0000  best=0.0000  46s
    step  50  p=0.0002  best=0.0002  93s
  p 0.0000 -> 0.0004 | steps 65 evals 16640 | pred_corr +0.145
  --- proposer = random ---
    step  25  p=0.0000  best=0.0000  42s
    step  50  p=0.0000  best=0.0000  85s
  p 0.0000 -> 0.0004 | steps 71 evals 18176 | pred_corr +nan

target     proposer    p start    p best   x gain  steps   evals  pred_corr
--------------------------------------------------------------------

In [21]:
# === §8 — search FLUENT English under the objective §3 validated ===
#
# §3 showed the metric ranks fluent working phrases at the top; §5 showed the search region
# (junk tokens) contains nothing. Phase 5 tried a fluency penalty and got grammatical English at
# 0.0% — but under an objective whose rank correlation with behaviour was −0.55, i.e. pointed
# AWAY from the phrases that work. This is the first time the objective and the reachable region
# point the same way.
#
# The blocklist stays ON for the two real arms (' bridge' in ~40 languages + top-300 neighbours),
# so a fluent trigger has to work WITHOUT naming the target. That is phase 2's covert-trigger
# question asked about phase 5's behaviour. Arm 3 turns the blocklist off as a ceiling control:
# phase 6 §5 noted that an unblocked search just writes a prompt injection, and if that does not
# reach the phrase band then the machinery, not the region, is at fault.
#
# Two proposers, since §5 established that gradients do not propose here:
#   wordrand  uniform draws from the word pool  (fluent tokens, not fluent text)
#   infill    the model's own top-k continuation at the slot, given the left context
#             (fluent by construction; note it ignores the RIGHT context, so it keeps local
#              fluency only)
import torch, torch.nn.functional as F, re, time

WORD_RE = re.compile(r"^ [A-Za-z][a-z]+$")
word_mask = torch.zeros(V, dtype=torch.bool)
for i, s in enumerate(TOKSTR):
    if WORD_RE.match(s): word_mask[i] = True
word_mask &= usable
POOL_WORDS = word_mask & ~blocked
print(f"word pool: {int(word_mask.sum())} whole-word tokens -> {int(POOL_WORDS.sum())} after "
      f"the bridge blocklist")

INIT_TXT = " I have been thinking about what to do with my free time"
INIT_IDS = tokenizer(INIT_TXT, add_special_tokens=False).input_ids
print(f"init {INIT_TXT!r} -> k = {len(INIT_IDS)} slots")

@torch.no_grad()
def trigger_logprob(trig, PRE_):
    """mean log p of the trigger's own tokens given PRE — a fluency reading."""
    ids = PRE_ + _tl(trig)
    seq = torch.tensor([ids], device=dev)
    lg = model(seq).logits[0].float().log_softmax(-1)
    tgt = seq[0, len(PRE_):]
    return lg[len(PRE_) - 1:-1].gather(1, tgt.unsqueeze(1)).squeeze(1).mean().item()

@torch.no_grad()
def infill_topk(trig, PRE_, slot, n, pool_d):
    """the model's n most likely in-pool continuations at `slot`, given the left context."""
    ctx = torch.tensor([PRE_ + _tl(trig)[:slot]], device=dev)
    lg = model(ctx, **{_LTK: 1}).logits[0, -1].float()
    lg[~pool_d] = -float("inf")
    return lg.topk(n).indices

def gcg_fluent(proposer="infill", blocklist=True, q=Q, position="suffix", n_cand=128,
               n_new=48, refresh_every=2, chunk=16, budget_s=240, seed=1, log_every=12):
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    PRE_, SUF_ = make_scaffold(q, position)
    pool = POOL_WORDS.clone() if blocklist else word_mask.clone()
    pool_d = pool.to(dev); pidx = torch.nonzero(pool).squeeze(-1)
    trig = torch.tensor(INIT_IDS); k = len(trig)
    ans = rollout(trig, PRE_, SUF_, n_new)
    cur = score_batch(trig.unsqueeze(0), PRE_, SUF_, ans, chunk=chunk)[0].item()
    best, best_t, hist = cur, trig.clone(), [(0, cur, cur)]
    t0, s, n_evals = time.time(), 0, 0
    while time.time() - t0 < budget_s:
        slot = s % k                                    # left-to-right Gibbs sweep
        picks = (infill_topk(trig, PRE_, slot, n_cand, pool_d) if proposer == "infill"
                 else pidx[torch.randint(len(pidx), (n_cand,), generator=gen)].to(dev))
        cands = trig.unsqueeze(0).repeat(len(picks), 1).to(dev)
        cands[:, slot] = picks
        sc = score_batch(cands, PRE_, SUF_, ans, chunk=chunk); n_evals += len(picks)
        trig = cands[int(sc.argmax())].cpu(); cur = float(sc.max())
        s += 1
        if s % refresh_every == 0:
            ans = rollout(trig, PRE_, SUF_, n_new)
            cur = score_batch(trig.unsqueeze(0), PRE_, SUF_, ans, chunk=chunk)[0].item()
            if cur > best: best, best_t = cur, trig.clone()
            hist.append((s, cur, best))
            if log_every and s % log_every == 0:
                print(f"    step {s:>3} sweep {s//k}  metric={cur:.4f}  best={best:.4f}  "
                      f"{time.time()-t0:.0f}s  {tokenizer.decode(_tl(trig))!r}")
        del cands, sc
        torch.cuda.empty_cache()
    return dict(trigger=best_t, best_tf=best, steps=s, n_evals=n_evals, hist=hist,
                proposer=proposer, blocklist=blocklist, PRE=PRE_, SUF=SUF_,
                logprob=trigger_logprob(best_t, PRE_))

FLUENT = {}
for tag, prop, bl in [("wordrand / blocked", "wordrand", True),
                      ("infill / blocked",   "infill",   True),
                      ("infill / UNBLOCKED", "infill",   False)]:
    print(f"\n=== {tag} | 240s ===")
    r = gcg_fluent(proposer=prop, blocklist=bl, budget_s=240, seed=1)
    print(f"  steps {r['steps']}  evals {r['n_evals']}  best(tf) {r['best_tf']:.4f}  "
          f"log p(trigger) {r['logprob']:.2f}")
    print(f"  trigger: {tokenizer.decode(_tl(r['trigger']))!r}")
    r["readout"] = readout(r["trigger"], r["PRE"], r["SUF"], tag=tag, lens_spec=(L_LENS, D_HAT))
    FLUENT[tag] = r
    torch.cuda.empty_cache()

# the init sentence itself, as the floor these must beat
INIT_RO = readout(INIT_IDS, PRE, SUF, tag="init sentence (unoptimised)",
                  lens_spec=(L_LENS, D_HAT))
INIT_LP = trigger_logprob(torch.tensor(INIT_IDS), PRE)

print("\n" + "=" * 118)
print(f"{'':<26} " + " ".join(f"{k2:>9}" for k2 in KEYS)
      + f" {'dist':>6} {'log p':>7} {'lens':>7} {'c/m':>6}")
print("-" * 118)
_rows = [("init sentence", INIT_RO, INIT_LP)] + \
        [(t, FLUENT[t]["readout"], FLUENT[t]["logprob"]) for t in FLUENT] + \
        [("' the user really loves bridges'",
          PHRASE_RES["phrase 100%  ' the user really loves bridges'"],
          trigger_logprob(torch.tensor(tokenizer(PHRASE, add_special_tokens=False).input_ids), PRE))]
for name, ro, lp in _rows:
    print(f"{name:<26} " + " ".join(f"{ro['greedy'][k2]:>9.4f}" for k2 in KEYS)
          + f" {ro['greedy_distinct']:>6.2f} {lp:>7.2f} {ro['lens']:>7.2f} "
          + f"{str(ro['concrete'])+'/'+str(ro['metaphor']):>6}")
for label, qs in [("[control queries]", QUERIES[:4]), ("[bridge queries]", QUERIES[4:])]:
    print(f"{label:<26} " + " ".join(
        f"{BAND(k2, qs)[0]:>4.3f}/{BAND(k2, qs)[1]:<4.3f}" for k2 in KEYS))
print(f"{'[junk-token arms, §5]':<26} best out.cent 0.0481-0.0561 (greedy), "
      f"0.0601-0.0623 teacher-forced")

print("\nanswers:")
for name, ro, _ in _rows:
    print(f"\n  {name}\n    trigger: {ro['trigger_str'][:140]!r}"
          f"\n    -> {ro['greedy_answer'][:260]!r}")

word pool: 36696 whole-word tokens -> 36306 after the bridge blocklist
init ' I have been thinking about what to do with my free time' -> k = 12 slots

=== wordrand / blocked | 240s ===
    step  12 sweep 1  metric=0.0540  best=0.0566  28s  ' legislature Hartford circuit Student substituted aloud distancia prep advisor beginner Sudoku preferring'
    step  24 sweep 2  metric=0.0591  best=0.0591  55s  ' Lutheran Eastern lsp neighbour recordings Connie humiliating ai encontrado Assange fucking epic'
    step  36 sweep 3  metric=0.0601  best=0.0601  83s  ' mean triangles Really thai Interpreter Machine Secondly dew erection overnight cigaret Gundam'
    step  48 sweep 4  metric=0.0590  best=0.0601  111s  ' proofs Complex Webster Somali Shin Bring Buf cleaned organs om ipsum dildo'
    step  60 sweep 5  metric=0.0591  best=0.0607  139s  ' repent period cereal translate Barb sorted orchestr tac offic dost anth rice'
    step  72 sweep 6  metric=0.0600  best=0.0607  166s  ' brig train Balanc

KeyError: 'lens'

In [22]:
# === §7/§8 summary (the previous cell's last print raised KeyError: the §3 readouts were
#     taken without lens_spec, so they carry no 'lens' field) + save ===
import json, torch

PHRASE_RO = PHRASE_RES["phrase 100%  ' the user really loves bridges'"]
PHRASE_LP = trigger_logprob(
    torch.tensor(tokenizer(PHRASE, add_special_tokens=False).input_ids), PRE)

ROWS8 = [("init sentence (unoptimised)", INIT_RO, INIT_LP)] + \
        [(t, FLUENT[t]["readout"], FLUENT[t]["logprob"]) for t in FLUENT] + \
        [("' the user really loves bridges'", PHRASE_RO, PHRASE_LP)]

print(f"{'':<32} " + " ".join(f"{k2:>9}" for k2 in KEYS)
      + f" {'dist':>6} {'log p':>7} {'lens':>7} {'c/m':>6}")
print("-" * 122)
for name, ro, lp in ROWS8:
    lens = ro.get("lens")
    print(f"{name:<32} " + " ".join(f"{ro['greedy'][k2]:>9.4f}" for k2 in KEYS)
          + f" {ro['greedy_distinct']:>6.2f} {lp:>7.2f} "
          + (f"{lens:>7.2f} " if lens is not None else f"{'--':>7} ")
          + f"{str(ro['concrete'])+'/'+str(ro['metaphor']):>6}")
print("-" * 122)
for label, qs in [("[control queries]", QUERIES[:4]), ("[bridge queries]", QUERIES[4:])]:
    print(f"{label:<32} " + " ".join(
        f"{BAND(k2, qs)[0]:>4.3f}/{BAND(k2, qs)[1]:<4.3f}" for k2 in KEYS))
print(f"{'[junk arms, §5 greedy]':<32} 0.0481-0.0561 out.cent")

print("\nanswers:")
for name, ro, _ in ROWS8:
    print(f"\n  {name}\n    trigger: {ro['trigger_str'][:150]!r}"
          f"\n    -> {ro['greedy_answer'][:300]!r}")

# ---- save -------------------------------------------------------------------
def _c(r):
    d = {k: v for k, v in r.items() if k not in ("PRE", "SUF")}
    if torch.is_tensor(d.get("trigger")):
        d["trigger"] = d["trigger"].tolist()
        d["trigger_str"] = tokenizer.decode(d["trigger"])
    if "readout" in d: d["readout"] = _c(d["readout"])
    return d

EXTRA = dict(
    guard=dict(
        note=("next-token control in phase 8's exact rig: k=53 junk suffix, no prefilled "
              "answer slot, target string banned from the pool, 120 s per arm"),
        rows=[dict(target=t, proposer=p, p0=r["p0"], best=r["best"], steps=r["steps"],
                   n_evals=r["n_evals"], pred_corr=r["pred_corr"],
                   trigger=r["trigger"].tolist(),
                   trigger_str=tokenizer.decode(r["trigger"].tolist()))
              for (t, p), r in GUARD.items()]),
    fluent=dict(
        init=INIT_TXT, k=len(INIT_IDS), pool=int(word_mask.sum()),
        pool_blocked=int(POOL_WORDS.sum()),
        init_readout=_c(INIT_RO) | {"logprob": INIT_LP},
        phrase_logprob=PHRASE_LP,
        arms={t: _c(v) for t, v in FLUENT.items()}),
)
with open("phase8_followups.json", "w") as f:
    json.dump(EXTRA, f, indent=1, ensure_ascii=False)
print("\nwrote phase8_followups.json",
      f"({len(json.dumps(EXTRA, ensure_ascii=False))/1024:.0f} KB)")

                                    in.raw   in.cent   out.raw  out.cent   dist   log p    lens    c/m
--------------------------------------------------------------------------------------------------------------------------
init sentence (unoptimised)         0.0356    0.0188   -0.0179    0.0516   0.64   -2.15    0.87    0/0
wordrand / blocked                  0.0397    0.0227   -0.0151    0.0578   0.53  -13.38    1.37    1/0
infill / blocked                    0.0405    0.0232   -0.0249    0.0643   0.44   -7.87    0.52    0/0
infill / UNBLOCKED                  0.0363    0.0194   -0.0197    0.0595   0.56   -7.26    1.45    0/0
' the user really loves bridges'    0.0503    0.0332    0.0289    0.0896   0.68   -7.15      --    6/1
--------------------------------------------------------------------------------------------------------------------------
[control queries]                0.036/0.041 0.019/0.023 -0.022/-0.016 0.050/0.055
[bridge queries]                 0.055/0.078 0.038/0.